# OpenNext Layout Planner — GPU Training (Google Colab)

Fine-tune **FLAN-T5-base** (~60–90 min on T4).

**If you get `torch has no attribute _utils` or `T5ForConditionalGeneration` import error:**  
Run the **Install** cell — it pins `transformers<5`. Then **Runtime → Restart session** and re-run from **Upload** (restart clears `/content`).

### Order
1. GPU check → Config → Mount Drive  
2. **Upload zip** → **Setup**  
3. **Install** (if import fails → Restart session → Upload + Setup again)  
4. Train

In [ ]:
# GPU check
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# "zip_local"  = upload via Colab left sidebar → /content/ (RECOMMENDED)
# "zip_drive"  = file on Google Drive at REPO_ZIP path
# "zip_upload" = browser popup (often blocked — avoid if upload fails)
SETUP_MODE = "zip_local"

REPO_ZIP = "/content/drive/MyDrive/training-colab-bundle.zip"
LOCAL_ZIP = "/content/training-colab-bundle.zip"

REPO_URL = "https://github.com/OpenNextCMS/openext.git"
REPO_BRANCH = "pr-khadijamama"
USE_GIT_TOKEN = False
GIT_TOKEN = ""

# Retrain settings (v2 — fixes 512-token truncation that broke JSON output)
EPOCHS = 15
BATCH_SIZE = 2       # T4 + max_target_length 896: use 2 (OOM at 8)
GRAD_ACCUM = 4       # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
EVAL_BATCH_SIZE = 1
MAX_TARGET_LENGTH = 896  # was 512 — 24% of labels were cut off
LR = 2e-4
FRESH_RUN = True   # pass --fresh to ignore partial checkpoints
DRIVE_SAVE_DIR = "/content/drive/MyDrive/opennext-layout-planner"

In [ ]:
# Mount Google Drive — only needed for SETUP_MODE == "zip_drive" or saving model to Drive
if SETUP_MODE == "zip_drive" or True:  # keep True if you want model backup on Drive
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Drive mount (zip_local mode)")

## Step A — Upload zip (run the cell below BEFORE Setup)

1. Run the **Upload zip** cell below
2. When you see **"Choose Files"**, click it and pick `training-colab-bundle.zip` from your PC  
   (`d:\Professional\ATPL\nextcmsmains\training-colab-bundle.zip`)
3. Wait until it prints `OK: /content/training-colab-bundle.zip`
4. Then run the **Setup** cell

**Already on Google Drive?** Upload zip to drive.google.com, run Mount Drive, then set `SETUP_MODE = "zip_drive"` and run Upload cell (it copies from Drive).

In [ ]:
# STEP A — Upload zip (run this BEFORE Setup)
from pathlib import Path
import shutil

LOCAL = Path(LOCAL_ZIP)
DRIVE_COPY = Path("/content/drive/MyDrive/training-colab-bundle.zip")

print("Zips in /content:", [p.name for p in Path("/content").glob("*.zip")])

if LOCAL.exists() and LOCAL.stat().st_size > 1000:
    print(f"OK: {LOCAL} ({LOCAL.stat().st_size // 1024} KB) — run Setup cell next")
elif SETUP_MODE == "zip_drive" and Path(REPO_ZIP).exists():
    shutil.copy(REPO_ZIP, LOCAL)
    print(f"OK: copied from Drive → {LOCAL}")
elif DRIVE_COPY.exists():
    shutil.copy(DRIVE_COPY, LOCAL)
    print(f"OK: copied {DRIVE_COPY} → {LOCAL}")
else:
    print("\n>>> Click 'Choose Files' below — select training-colab-bundle.zip <<<\n")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(
            "No file selected.\n"
            "Click 'Choose Files' above, OR upload to drive.google.com and set SETUP_MODE='zip_drive'"
        )
    src = Path("/content") / next(iter(uploaded))
    if src.resolve() != LOCAL.resolve():
        shutil.copy(src, LOCAL)
    print(f"OK: saved to {LOCAL} ({LOCAL.stat().st_size // 1024} KB)")
    print("Now run the Setup cell below.")

In [ ]:
# STEP B — Setup (extract zip) — run AFTER Upload cell shows OK
import shutil, zipfile
from pathlib import Path

REPO_DIR = Path("/content/nextcmsmains")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir(parents=True)

zip_path = Path(LOCAL_ZIP)
if not zip_path.exists():
    raise FileNotFoundError(
        f"Missing {zip_path}\n"
        "Run the Upload cell above first and wait for OK message."
    )

print("Extracting", zip_path, f"({zip_path.stat().st_size // 1024} KB)")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall("/content")
if not (REPO_DIR / "training").exists() and Path("/content/training").exists():
    shutil.move("/content/training", str(REPO_DIR / "training"))

train_file = REPO_DIR / "training" / "data" / "seq2seq" / "train_seq2seq.jsonl"
assert train_file.exists(), "Bad zip file — re-run package_for_colab.py on PC"
print("Ready:", REPO_DIR)
print("Train rows:", sum(1 for _ in open(train_file, encoding="utf-8")))

In [ ]:
# Install dependencies (Colab already has CUDA torch — do NOT pip install torch)
# Pin transformers 4.x — v5 breaks Colab torch. Gradio/huggingface-hub warning below is OK to ignore.
%pip install -q "transformers==4.46.3" "datasets>=2.19" "accelerate>=0.30" "huggingface-hub>=0.23.2,<1.0" sentencepiece scikit-learn json-repair python-dotenv httpx

import torch
from transformers import T5ForConditionalGeneration
print("torch:", torch.__version__)
print("T5 import OK — ignore any gradio/huggingface-hub pip WARNING above")

In [ ]:
# Dataset sanity check
import json
from pathlib import Path

SEQ = REPO_DIR / "training" / "data" / "seq2seq"
for name in ("train", "eval", "test"):
    p = SEQ / f"{name}_seq2seq.jsonl"
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{name}: {n} rows")

In [ ]:
# Train on GPU (~45-90 min for 15 epochs on T4 with max-target-length 896)
import gc
import os
import time

import torch

os.chdir(REPO_DIR)

if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()

OUT = REPO_DIR / "training" / "layout-planner" / "outputs" / "layout-planner-flan-t5-base"
OUT.mkdir(parents=True, exist_ok=True)

fresh_flag = " --fresh" if FRESH_RUN else ""
cmd = (
    f"python training/layout-planner/train.py "
    f"--epochs {EPOCHS} "
    f"--batch-size {BATCH_SIZE} "
    f"--eval-batch-size {EVAL_BATCH_SIZE} "
    f"--grad-accum {GRAD_ACCUM} "
    f"--max-target-length {MAX_TARGET_LENGTH} "
    f"--lr {LR} "
    f"--output-dir {OUT}"
    f"{fresh_flag}"
)
print(cmd)
t0 = time.time()
!{cmd}
print(f"\nTraining finished in {(time.time()-t0)/60:.1f} minutes")

In [ ]:
# Verify model weights saved
from pathlib import Path
weights = list(OUT.glob("model.safetensors")) + list(OUT.glob("*.bin"))
assert weights, f"No model weights in {OUT} — training may have failed"
print("Model files:")
for f in sorted(OUT.iterdir()):
  if f.is_file():
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Copy trained model to Google Drive (persistent after Colab disconnects)
import shutil
from pathlib import Path
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d-%H%M")
drive_dest = Path(DRIVE_SAVE_DIR) / f"layout-planner-flan-t5-base-{stamp}"
if drive_dest.exists():
    shutil.rmtree(drive_dest)
shutil.copytree(OUT, drive_dest)
print("Saved to Drive:", drive_dest)

In [ ]:
# Quick inference test — model ONLY (no retrieval fallback)
import sys
sys.path.insert(0, str(REPO_DIR / "training" / "layout-planner"))
from lib.inference import LayoutPlannerModel
import json

planner = LayoutPlannerModel(model_dir=OUT)
prompt = "Create a premium dental clinic website in Ahmedabad."
result = planner.plan_layout(prompt, page_name="Home", max_attempts=2, use_fallback=False)
print("valid:", result["valid"], "| source:", result.get("source"))
print("errors:", result.get("errors"))
if result.get("layoutPlan"):
    lp = result["layoutPlan"]
    print("meta:", json.dumps(lp.get("meta", {}), indent=2))
    blocks = lp.get("page", {}).get("blocks", [])
    print("blocks:", [b.get("type") for b in blocks])
if not result["valid"]:
    print("RAW:", (result.get("raw") or "")[:300])

In [ ]:
# Zip for download to your PC
import shutil
zip_path = "/content/layout-planner-model.zip"
shutil.make_archive("/content/layout-planner-model", "zip", OUT)
print("Download this file from Colab file browser:", zip_path)
from google.colab import files
files.download(zip_path)

## Use the model on your PC

1. Download the zip from Colab **or** copy from Google Drive
2. Extract to:
   `training/layout-planner/outputs/layout-planner-flan-t5-base/`
3. Run:
```bash
python training/layout-planner/pipeline.py --layout-only --prompt "Create a premium dental clinic website in Ahmedabad." --print-json
```

## Resume training in Colab (if disconnected)

Set `RESUME_CHECKPOINT` to the latest `checkpoint-*` folder inside `OUT`, then re-run the train cell with:
```python
!python training/layout-planner/train.py --epochs 10 --batch-size 8 --resume-from {latest_checkpoint}
```